# USGS Streamflow → tsl Pipeline

Builds the two objects needed to drop into `SpatioTemporalDataset`:
1. **`target`** — `(T, N)` DataFrame of daily discharge (m³/s), columns = `site0`…`site30`
2. **`connectivity`** — `(edge_index, edge_weight)` in COO format from the watershed directed graph

Graph topology comes from `02_Generate_Graph/generate_graph.ipynb`.  
Edges point **upstream → downstream** (physical flow direction).

## 1. Load and prepare target DataFrame

In [1]:
import os
import pandas as pd
import numpy as np
import torch

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/clean/'))

# Load discharge time series — columns are USGS site IDs, rows are daily timesteps
streamflow = pd.read_csv(
    os.path.join(DATA_DIR, 'streamflow_wy2022.csv'),
    dtype=str,          # read all as str to preserve leading zeros in site IDs
    parse_dates=['date'],
    index_col='date'
).astype(float)         # convert values back to float after parsing

print(f'Raw shape: {streamflow.shape}  ({streamflow.shape[0]} days, {streamflow.shape[1]} gauges)')
print('Date range:', streamflow.index[0].date(), '->', streamflow.index[-1].date())
streamflow.head(3)

Raw shape: (365, 31)  (365 days, 31 gauges)
Date range: 2021-10-01 -> 2022-09-30


,01413088,01413398,01413408,01413500,01414000,01414500,01415000,01415460,01417000,01417500,...,01426500,01427207,01427500,01427510,01428000,01428500,01422779,01425805,01427195,01424998
date,,,,,,,,,,,,,,,,,,,,,
2021-10-01,0.39903,1.86214,4.0752,8.6032,1.09521,1.25935,1.34142,0.277623,19.3572,23.9701,...,46.978,95.937,5.9713,106.691,1.72347,119.426,0.94805,0.83202,1.78007,NaN
2021-10-02,0.36790,1.66970,3.6507,7.8391,0.99616,1.14332,1.18011,0.248757,19.3572,23.4890,...,46.695,92.258,5.1789,101.031,1.51971,113.200,0.84900,0.74995,1.56499,NaN
2021-10-03,0.36507,1.54518,3.3677,7.3863,0.94522,1.04993,1.11502,0.231494,19.3572,23.2060,...,46.695,89.711,4.7261,97.069,1.41783,108.955,0.79806,0.69901,1.39236,NaN


In [2]:
# Site rename mapping — same order as in generate_graph.ipynb
# CSV columns follow the metadata row order, NOT numeric site order
site_rename_list = [
    'site0',  'site1',  'site2',  'site3',  'site4',
    'site6',  'site7',  'site9',  'site11', 'site13',
    'site14', 'site15', 'site16', 'site17', 'site5',
    'site10', 'site12', 'site8',
    'site18', 'site19', 'site20', 'site21', 'site22',
    'site23', 'site24', 'site25', 'site26', 'site27',
    'site28', 'site29', 'site30'
]

rename_map = dict(zip(streamflow.columns, site_rename_list))
target = streamflow.rename(columns=rename_map)

# Sort columns site0 → site30 so node index i == int(sitei)
target = target[sorted(target.columns, key=lambda x: int(x.replace('site', '')))]

print(f'Target shape: {target.shape}  — ready for SpatioTemporalDataset')
print('Columns:', list(target.columns))
target.head(3)

Target shape: (365, 31)  — ready for SpatioTemporalDataset
Columns: ['site0', 'site1', 'site2', 'site3', 'site4', 'site5', 'site6', 'site7', 'site8', 'site9', 'site10', 'site11', 'site12', 'site13', 'site14', 'site15', 'site16', 'site17', 'site18', 'site19', 'site20', 'site21', 'site22', 'site23', 'site24', 'site25', 'site26', 'site27', 'site28', 'site29', 'site30']


,site0,site1,site2,site3,site4,site5,site6,site7,site8,site9,...,site21,site22,site23,site24,site25,site26,site27,site28,site29,site30
date,,,,,,,,,,,,,,,,,,,,,
2021-10-01,0.39903,1.86214,4.0752,8.6032,1.09521,0.59713,1.25935,1.34142,0.65656,0.277623,...,46.978,95.937,5.9713,106.691,1.72347,119.426,0.94805,0.83202,1.78007,NaN
2021-10-02,0.36790,1.66970,3.6507,7.8391,0.99616,0.52921,1.14332,1.18011,0.58298,0.248757,...,46.695,92.258,5.1789,101.031,1.51971,113.200,0.84900,0.74995,1.56499,NaN
2021-10-03,0.36507,1.54518,3.3677,7.3863,0.94522,0.58581,1.04993,1.11502,0.53770,0.231494,...,46.695,89.711,4.7261,97.069,1.41783,108.955,0.79806,0.69901,1.39236,NaN


In [3]:
# Quick sanity check — any NaNs?
nan_counts = target.isna().sum()
print('NaN counts per gauge:')
print(nan_counts[nan_counts > 0] if nan_counts.any() else 'None — all gauges complete')

NaN counts per gauge:
site4       1
site9      27
site14     36
site22     37
site23      5
site24     26
site25      3
site26     28
site30    279
dtype: int64


## 2. Build connectivity (edge_index + edge_weight)

Topology is taken directly from `generate_graph.ipynb`.  
Edges are **directed upstream → downstream**.  
Weights are set to 1 (binary adjacency); swap in distance or drainage-area weights below if needed.

In [4]:
# Directed edge list — upstream node first, downstream node second
graph_edges_all = [
    # East Branch subbasin (sites 0-17)
    ('site0',  'site2'),
    ('site1',  'site2'),
    ('site2',  'site6'),
    ('site3',  'site6'),
    ('site4',  'site6'),
    ('site5',  'site6'),
    ('site6',  'site8'),   # both site6 and site7 feed the reservoir
    ('site7',  'site8'),   # site8 is the post-reservoir gauge
    ('site8',  'site9'),
    ('site9',  'site11'),
    ('site10', 'site11'),
    ('site11', 'site13'),
    ('site12', 'site13'),
    ('site13', 'site15'),
    ('site14', 'site15'),
    ('site15', 'site17'),
    ('site16', 'site17'),
    # West Branch subbasin (sites 18-30)
    ('site18', 'site21'),
    ('site19', 'site20'),
    ('site20', 'site21'),
    ('site21', 'site26'),  # site26 = outlet of Pepacton Reservoir
    ('site22', 'site26'),
    ('site23', 'site26'),
    ('site24', 'site26'),
    ('site25', 'site26'),
    ('site26', 'site27'),
    ('site27', 'site30'),
    ('site28', 'site29'),
    ('site29', 'site30'),
    # Junction connecting the two subbasins
    ('site30', 'site13'),
]

# Node index = integer suffix of site name (site0 -> 0, site13 -> 13, etc.)
node_to_idx = {f'site{i}': i for i in range(31)}

src = [node_to_idx[u] for u, v in graph_edges_all]
dst = [node_to_idx[v] for u, v in graph_edges_all]

edge_index  = torch.tensor([src, dst], dtype=torch.long)
edge_weight = torch.ones(edge_index.shape[1], dtype=torch.float)

connectivity = (edge_index, edge_weight)

print(f'Nodes : {len(node_to_idx)}')
print(f'Edges : {edge_index.shape[1]}')
print(f'edge_index shape : {edge_index.shape}  <- (2, E): row0=src (upstream), row1=dst (downstream)')
print(f'edge_weight range: [{edge_weight.min():.2f}, {edge_weight.max():.2f}]')

Nodes : 31
Edges : 30
edge_index shape : torch.Size([2, 30])  <- (2, E): row0=src (upstream), row1=dst (downstream)
edge_weight range: [1.00, 1.00]


### Optional: distance-based edge weights

Replace binary weights with a Gaussian kernel on lat/lon distance between gauges —
same approach MetrLA uses for road distances.

In [5]:
metadata = pd.read_csv(
    os.path.join(DATA_DIR, 'streamflow_wy2022_metadata_all.csv'),
    dtype={'site_id': str, 'huc8': str}
)
metadata['site_rename'] = site_rename_list
metadata = metadata.set_index('site_rename').loc[target.columns]  # align to site0..site30 order

coords = metadata[['latitude', 'longitude']].values  # (N, 2)

# Haversine-like approximation: Euclidean on lat/lon degrees (fine for a ~200km basin)
def gaussian_edge_weights(src_idx, dst_idx, coords, sigma=0.5):
    """Gaussian kernel on Euclidean lat/lon distance."""
    d = np.linalg.norm(coords[src_idx] - coords[dst_idx], axis=1)
    return np.exp(-d**2 / (2 * sigma**2))

w = gaussian_edge_weights(src, dst, coords, sigma=0.5)
edge_weight_dist = torch.tensor(w, dtype=torch.float)

connectivity_dist = (edge_index, edge_weight_dist)

print(f'Distance-based weight range: [{edge_weight_dist.min():.3f}, {edge_weight_dist.max():.3f}]')

Distance-based weight range: [0.393, 0.999]


## 3. Plug into SpatioTemporalDataset

With `target` and `connectivity` ready, the rest follows the MetrLA pattern from `TotchSpatiotemporal_trial.ipynb`.

In [6]:
from tsl.data import SpatioTemporalDataset

WINDOW  = 7   # 7 days of past discharge as input
HORIZON = 3   # predict the next 3 days

torch_dataset = SpatioTemporalDataset(
    target=target,
    connectivity=connectivity,   # swap to connectivity_dist for weighted edges
    window=WINDOW,
    horizon=HORIZON,
    stride=1,
)

print('Number of samples:', len(torch_dataset))

sample = torch_dataset[0]
print('\nInput  x : (window, N, C) =', sample.input.x.shape)
print('Target y : (horizon, N, C) =', sample.y.shape)

/opt/anaconda3/envs/tsl_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of samples: 356

Input  x : (window, N, C) = torch.Size([7, 31, 1])
Target y : (horizon, N, C) = torch.Size([3, 31, 1])
